In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.default.urbanpulse_bronze;


In [0]:
bronze_path = "/Volumes/workspace/default/urbanpulse_bronze"
files = dbutils.fs.ls(bronze_path)
for file in files:
    print(file.name, file.size)


In [0]:
weather_path = (
    "/Volumes/workspace/default/urbanpulse_bronze/"
    "delhi_weather_2024-08-11_to_2024-08-30.json"
)
weather_df = spark.read.option("multiLine", "true").json(weather_path)
display(weather_df.limit(10))


In [0]:

from pyspark.sql import functions as F

# 1. Check total number of records
print("Total records:", weather_df.count())

# 2. Check column names
print("Columns:", weather_df.columns)

# 3. Check data types
weather_df.printSchema()

# 4. Check missing values in each column
null_counts = weather_df.select([
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c)
    for c in weather_df.columns
])

display(null_counts)

# 5. View sample records
display(weather_df.limit(10))


In [0]:

from pyspark.sql import functions as F

# Inspect all fields inside the hourly structure
weather_df.select("hourly.*").printSchema()

# Check the number of values in each hourly array
array_lengths = weather_df.select(
    F.size("hourly.time").alias("time_count"),
    F.size("hourly.temperature_2m").alias("temperature_count"),
    F.size("hourly.relative_humidity_2m").alias("humidity_count"),
    F.size("hourly.precipitation").alias("precipitation_count"),
    F.size("hourly.rain").alias("rain_count"),
    F.size("hourly.weather_code").alias("weather_code_count")
)

display(array_lengths)


In [0]:

from pyspark.sql import functions as F

# Combine all hourly arrays into one array of structured records
weather_flat = weather_df.select(
    F.col("latitude"),
    F.col("longitude"),
    F.col("elevation"),
    F.explode(
        F.arrays_zip(
            F.col("hourly.time"),
            F.col("hourly.temperature_2m"),
            F.col("hourly.relative_humidity_2m"),
            F.col("hourly.precipitation"),
            F.col("hourly.rain"),
            F.col("hourly.weather_code"),
            F.col("hourly.cloud_cover"),
            F.col("hourly.wind_speed_10m"),
            F.col("hourly.wind_direction_10m")
        )
    ).alias("hourly_record")
)

# Extract individual values into columns
weather_flat = weather_flat.select(
    "latitude",
    "longitude",
    "elevation",
    F.col("hourly_record.time").alias("time"),
    F.col("hourly_record.temperature_2m").alias("temperature_2m"),
    F.col("hourly_record.relative_humidity_2m").alias("relative_humidity_2m"),
    F.col("hourly_record.precipitation").alias("precipitation"),
    F.col("hourly_record.rain").alias("rain"),
    F.col("hourly_record.weather_code").alias("weather_code"),
    F.col("hourly_record.cloud_cover").alias("cloud_cover"),
    F.col("hourly_record.wind_speed_10m").alias("wind_speed_10m"),
    F.col("hourly_record.wind_direction_10m").alias("wind_direction_10m")
)

display(weather_flat.limit(10))


In [0]:
print("Flattened records:", weather_flat.count())
weather_flat.printSchema()


In [0]:

from pyspark.sql import functions as F

# Convert time into Spark timestamp
weather_silver = weather_flat.withColumn(
    "time",
    F.to_timestamp("time", "yyyy-MM-dd'T'HH:mm")
)

# Remove records without a valid timestamp
weather_silver = weather_silver.filter(
    F.col("time").isNotNull()
)

# Remove duplicate hourly observations
weather_silver = weather_silver.dropDuplicates(["time"])

# Add date and hour columns for analytics
weather_silver = (
    weather_silver
    .withColumn("weather_date", F.to_date("time"))
    .withColumn("weather_hour", F.hour("time"))
)

# Check the cleaned DataFrame
print("Silver record count:", weather_silver.count())

display(weather_silver.limit(10))



In [0]:

# Check missing values
display(
    weather_silver.select([
        F.count(
            F.when(F.col(c).isNull(), c)
        ).alias(c)
        for c in weather_silver.columns
    ])
)

# Check temperature range
display(
    weather_silver.select(
        F.min("temperature_2m").alias("min_temperature"),
        F.max("temperature_2m").alias("max_temperature"),
        F.avg("temperature_2m").alias("avg_temperature")
    )
)

# Check duplicate timestamps
print(
    "Duplicate timestamps:",
    weather_silver.count() -
    weather_silver.select("time").distinct().count()
)


In [0]:

silver_table = "workspace.default.silver_weather"

(
    weather_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table)
)

print("Silver table created:", silver_table)

